In [1]:
import climt
from sympl import DataArray, TendencyComponent, AdamsBashforth
from sympl import (
    PlotFunctionMonitor, NetCDFMonitor,
    TimeDifferencingWrapper, UpdateFrequencyWrapper,
    set_constant, get_constant, initialize_numpy_arrays_with_properties
)
import gfs_dynamical_core
from datetime import timedelta
import numpy as np
import torch
import torch.nn as nn
import sys
sys.path.append("..")
from models import DynamicMLP, DynamicMLP_flatten

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/projects/sds-lab/Shuochen/miniconda3/envs/ai/lib/python3.9/site-packages/climt/_core/initialization.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
class TorchScaler:
    def __init__(self, mean, std):
        self.mean = mean  # [C]
        self.std = std
    
    def transform(self, x):
        mean = self.mean
        std = self.std
        if isinstance(mean, torch.Tensor):
            mean = mean.cpu().numpy()
        if isinstance(std, torch.Tensor):
            std = std.cpu().numpy()
        return (x - mean) / std
    
    def inverse_transform(self, x):
        mean = self.mean
        std = self.std
        if isinstance(mean, torch.Tensor):
            mean = mean.cpu().numpy()
        if isinstance(std, torch.Tensor):
            std = std.cpu().numpy()
        return x * std + mean

class NNParameterization(TendencyComponent):

    input_properties = {
        'air_temperature': {'dims': ['*', 'mid_levels'], 'units': 'degK'},
        'specific_humidity': {'dims': ['*', 'mid_levels'], 'units': 'kg/kg'},
        'surface_air_pressure': {'dims': ['*'], 'units': 'Pa'},
        'surface_upward_latent_heat_flux': {'dims': ['*'], 'units': 'W m^-2'},
        'surface_upward_sensible_heat_flux': {'dims': ['*'], 'units': 'W m^-2'},
    }

    tendency_properties = {
        'air_temperature': {
            'units': 'degK day^-1'
        }
    }
    
    diagnostic_properties = {}

    def __init__(self, **kwargs):
        super(NNParameterization, self).__init__(**kwargs)

        self.IN_FEATURES = 59
        self.OUT_FEATURES = 28

        ckpt = torch.load("./best_model/best_model_trial_0.pth",map_location=device)
        ckpt_norm = torch.load('/projects/sds-lab/Shuochen/climt/gmd_aquaplanet/64x32_normalization.pth', map_location=device)
        self.input_scaler  = TorchScaler(ckpt_norm['X_mean'], ckpt_norm['X_std'])
        self.output_scaler = TorchScaler(ckpt_norm['y_mean'], ckpt_norm['y_std'])

        self.model = DynamicMLP_flatten(self.IN_FEATURES,self.OUT_FEATURES,ckpt["hidden_sizes"]).to(device)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()

    def array_call(self, state):

        num_cols, num_levs = state['air_temperature'].shape
        
        tendencies = initialize_numpy_arrays_with_properties(
            self.tendency_properties, state, self.input_properties
        )
        diagnostics = initialize_numpy_arrays_with_properties(
            self.diagnostic_properties, state, self.input_properties
        )
        # ---------------------------------
        # Extract state
        # ---------------------------------
        T = state['air_temperature']        # ()
        q = state['specific_humidity']      # ()
        ps = state['surface_air_pressure']  # ()
        lh = state['surface_upward_latent_heat_flux'] # ()
        sh = state['surface_upward_sensible_heat_flux'] # ()
        # ---------------------------
        # Build NN input []
        # ---------------------------        
        x = np.concatenate([
            T,
            q,
            ps[..., None],
            lh[..., None],
            sh[..., None],
        ], axis=-1)  # ()
        
        # normalize
        if self.input_scaler is not None:
            x = self.input_scaler.transform(x)
            
        # Add batch dimension
        x = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)
        # ---------------------------------
        # NN inference
        # ---------------------------------
        with torch.no_grad():
            y = self.model(x)   # ()
        y = y.cpu().numpy().squeeze()
        # y = y / 86400.0
        # print(y.shape)
        # inverse normalize
        if self.output_scaler is not None:
            y = self.output_scaler.inverse_transform(y)
            
        # ---------------------------------
        # Map NN output → temperature tendency
        # ---------------------------------
        tendencies['air_temperature'][:] = y[:, :num_levs] #shape -> lon*lat, cols

        return tendencies, diagnostics
        
set_constant('stellar_irradiance', value=200, units='W m^-2')
model_time_step = timedelta(minutes=10)
# Create components
convection = climt.EmanuelConvection(tendencies_in_diagnostics=True)
simple_physics = TimeDifferencingWrapper(climt.SimplePhysics())
radiation_step = timedelta(hours=1)
# radiation_lw = UpdateFrequencyWrapper(climt.RRTMGLongwave(), radiation_step)
radiation_sw = UpdateFrequencyWrapper(climt.RRTMGShortwave(), radiation_step)
slab_surface = climt.SlabSurface()
# nn component
nn_component = NNParameterization()
dycore = gfs_dynamical_core.GFSDynamicalCore([simple_physics, slab_surface, radiation_sw, nn_component, convection], number_of_damped_levels=5)
# dycore = gfs_dynamical_core.GFSDynamicalCore([simple_physics, slab_surface, radiation_sw, radiation_lw, convection], number_of_damped_levels=5)
grid = climt.get_grid(nx=64, ny=32)

my_state = climt.get_default_state([dycore], grid_state=grid)
# Set initial/boundary conditions
latitudes = my_state['latitude'].values
longitudes = my_state['longitude'].values
zenith_angle = np.radians(latitudes)
surface_shape = latitudes.shape
my_state['zenith_angle'].values = zenith_angle
my_state['eastward_wind'].values[:] = np.random.randn(
    *my_state['eastward_wind'].shape)
my_state['ocean_mixed_layer_thickness'].values[:] = 10
surf_temp_profile = 290 - (40*np.sin(zenith_angle)**2)
my_state['surface_temperature'].values = surf_temp_profile

for i in range(10000):
    # print(my_state.keys())
    diag, my_state = dycore(my_state, model_time_step)
    my_state.update(diag)
    my_state['time'] += model_time_step

    if i % 100 == 0:
        T_mid = my_state['air_temperature'].values[15].mean()
        print(f"Step {i}, mean T_mid = {T_mid:.2f} K")

Step 0, mean T_mid = 290.00 K
Step 100, mean T_mid = 290.14 K
Step 200, mean T_mid = 290.25 K
Step 300, mean T_mid = 290.35 K
Step 400, mean T_mid = 290.43 K
Step 500, mean T_mid = 290.51 K
Step 600, mean T_mid = 290.57 K
Step 700, mean T_mid = 290.58 K
Step 800, mean T_mid = 290.53 K
Step 900, mean T_mid = 290.45 K
Step 1000, mean T_mid = 290.28 K
Step 1100, mean T_mid = 290.00 K
Step 1200, mean T_mid = 284.82 K
Step 1300, mean T_mid = 269.95 K
Step 1400, mean T_mid = 268.20 K
Step 1500, mean T_mid = 267.09 K
Step 1600, mean T_mid = 266.35 K
Step 1700, mean T_mid = 265.62 K
Step 1800, mean T_mid = 264.42 K
Step 1900, mean T_mid = 264.16 K
Step 2000, mean T_mid = 264.31 K
Step 2100, mean T_mid = 264.34 K
Step 2200, mean T_mid = 264.37 K
Step 2300, mean T_mid = 264.56 K
Step 2400, mean T_mid = 262.27 K
Step 2500, mean T_mid = 262.80 K
